# 04_train_transformer_intent
End-to-end transformer training notebook for intent classification.

This notebook assumes `data/processed/conversation_dataset.csv` already exists and calls the training script.

In [1]:
# If you need them, install dependencies locally:
# !pip install -U transformers datasets accelerate scikit-learn pandas torch joblib
print("Ready to run.")

Ready to run.


In [2]:
from pathlib import Path
import pandas as pd

csv_path = Path("../data/processed/conversation_dataset.csv")
print("CSV exists:", csv_path.exists(), csv_path)
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df.head(3))
    print("Rows:", len(df))
    print("Intent distribution:")
    print(df["gt_primary_intent"].value_counts())

CSV exists: True ..\data\processed\conversation_dataset.csv


,conversation_id,conversation_text,user_text,gt_primary_intent,gt_authenticated,gt_tool_failure,gt_sentiment_overall,gt_turn_count,events_json
0,55be8f45-1846-482c-8b81-79ae83716f39@0.0.0.0,USER: Something seems off maybe with my accoun...,Something seems off maybe with my account. The...,CARD_REPLACEMENT,NaN,False,-0.3339,7,"[{""event_id"": ""548d0604-bf02-4c37-9013-5a4a9b0..."
1,9ae8922a-4cb3-439c-9d43-aa98feb4d961@0.0.0.0,USER: I have a question about my account. I ca...,I have a question about my account. I can't ma...,CARD_REPLACEMENT,NaN,True,-0.2542,11,"[{""event_id"": ""72d79b73-db0e-4c32-9660-96e29b9..."
2,f65977c7-251c-4ec5-99ea-e3ccbbc5e897@0.0.0.0,USER: I need help with a couple of things. I t...,I need help with a couple of things. I think a...,TRANSACTION_DISPUTE,NaN,False,0.1133,12,"[{""event_id"": ""25789307-92b8-45d8-8a53-76babad..."


Rows: 10000
Intent distribution:
gt_primary_intent
PAYMENT_DUE_DATE       2044
ADDRESS_UPDATE         2015
CARD_REPLACEMENT       1999
TRANSACTION_DISPUTE    1994
PAYMENT_ISSUE          1948
Name: count, dtype: int64


In [7]:

# Train the transformer baseline (distilbert-base-uncased by default)
# For a quicker test, you can set --epochs 1 first.
!python ../src/train_transformer_intent.py \
  --csv ../data/processed/conversation_dataset.csv \
  --output-dir ../models/transformer_intent \
  --model-name distilbert-base-uncased \
  --epochs 3 \
  --max-length 256 \
  --train-batch-size 8 \
  --eval-batch-size 16 \
  --learning-rate 2e-5 \
  --overwrite-output-dir

Starting training...
{'loss': '1.615', 'grad_norm': '1.794', 'learning_rate': '1.982e-05', 'epoch': '0.02857'}
{'loss': '1.503', 'grad_norm': '4.707', 'learning_rate': '1.963e-05', 'epoch': '0.05714'}
{'loss': '0.7786', 'grad_norm': '4.563', 'learning_rate': '1.944e-05', 'epoch': '0.08571'}
{'loss': '0.2453', 'grad_norm': '1.183', 'learning_rate': '1.925e-05', 'epoch': '0.1143'}
{'loss': '0.0943', 'grad_norm': '0.4988', 'learning_rate': '1.906e-05', 'epoch': '0.1429'}
{'loss': '0.03864', 'grad_norm': '0.4013', 'learning_rate': '1.886e-05', 'epoch': '0.1714'}
{'loss': '0.02459', 'grad_norm': '0.2201', 'learning_rate': '1.867e-05', 'epoch': '0.2'}
{'loss': '0.01665', 'grad_norm': '0.1235', 'learning_rate': '1.848e-05', 'epoch': '0.2286'}
{'loss': '0.01449', 'grad_norm': '0.1075', 'learning_rate': '1.829e-05', 'epoch': '0.2571'}
{'loss': '0.01116', 'grad_norm': '0.1119', 'learning_rate': '1.81e-05', 'epoch': '0.2857'}
{'loss': '0.008669', 'grad_norm': '0.08111', 'learning_rate': '1.791e-0


Map: 100%|██████████| 7000/7000 [00:08<00:00, 842.19 examples/s]

Map: 100%|██████████| 1000/1000 [00:01<00:00, 780.50 examples/s]

Map: 100%|██████████| 2000/2000 [00:02<00:00, 854.37 examples/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 657.06it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those para

In [8]:
# Inspect outputs
from pathlib import Path
import pandas as pd

out_dir = Path("../models/transformer_intent")
print("Output dir exists:", out_dir.exists())

for name in ["dataset_split_counts.json", "training_summary.json", "label_maps.json", "validation_metrics.json", "test_metrics.json"]:
    p = out_dir / name
    print(f"{name}: {p.exists()}")
    if p.exists():
        print(p.read_text()[:1000])
        print("-" * 60)

test_pred = out_dir / "test_predictions.csv"
if test_pred.exists():
    preds = pd.read_csv(test_pred)
    display(preds.head(10))
    print("Prediction rows:", len(preds))

Output dir exists: True
dataset_split_counts.json: True
{
  "train": 7000,
  "validation": 1000,
  "test": 2000,
  "total": 10000
}
------------------------------------------------------------
training_summary.json: True
{
  "model_name": "distilbert-base-uncased",
  "max_length": 256,
  "epochs": 3,
  "train_batch_size": 8,
  "eval_batch_size": 16,
  "learning_rate": 2e-05,
  "validation": {
    "accuracy": 1.0,
    "macro_f1": 1.0,
    "weighted_f1": 1.0
  },
  "test": {
    "accuracy": 1.0,
    "macro_f1": 1.0,
    "weighted_f1": 1.0
  }
}
------------------------------------------------------------
label_maps.json: True
{
  "label2id": {
    "ADDRESS_UPDATE": 0,
    "CARD_REPLACEMENT": 1,
    "PAYMENT_DUE_DATE": 2,
    "PAYMENT_ISSUE": 3,
    "TRANSACTION_DISPUTE": 4
  },
  "id2label": {
    "0": "ADDRESS_UPDATE",
    "1": "CARD_REPLACEMENT",
    "2": "PAYMENT_DUE_DATE",
    "3": "PAYMENT_ISSUE",
    "4": "TRANSACTION_DISPUTE"
  }
}
-------------------------------------------------

,label_id,pred_id,label,pred
0,2,2,PAYMENT_DUE_DATE,PAYMENT_DUE_DATE
1,3,3,PAYMENT_ISSUE,PAYMENT_ISSUE
2,3,3,PAYMENT_ISSUE,PAYMENT_ISSUE
3,3,3,PAYMENT_ISSUE,PAYMENT_ISSUE
4,1,1,CARD_REPLACEMENT,CARD_REPLACEMENT
5,3,3,PAYMENT_ISSUE,PAYMENT_ISSUE
6,1,1,CARD_REPLACEMENT,CARD_REPLACEMENT
7,2,2,PAYMENT_DUE_DATE,PAYMENT_DUE_DATE
8,2,2,PAYMENT_DUE_DATE,PAYMENT_DUE_DATE
9,0,0,ADDRESS_UPDATE,ADDRESS_UPDATE


Prediction rows: 2000
